# RNN Baseline — Mitt (v2: class-weighted loss)

**What changed from the first run:** the first version of this notebook collapsed to majority-class
guessing on both tasks (always predicted InSeq, always predicted odor A), because the loss function had
no penalty for getting the rare classes wrong. The GLM baseline avoided this using
`class_weight='balanced'`. This version adds the equivalent fix to the RNN: per-class weights, computed
from the training set only, passed into `CrossEntropyLoss` for both heads.

Everything else (data loading, windowing, model architecture, train/val/test split) is unchanged from
the first version, this is a targeted fix, not a rebuild.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix

from src.preprocessing import build_labels, segment_trials, get_sampling_rate
from src.models import MultiTaskRNN
from src.utils import set_seed, load_config, get_device


## 1. Setup: config, seed, device

In [ ]:
cfg = load_config('../configs/baseline.yaml')
set_seed(cfg['seed'])
device = get_device()


## 2. Load data and build labels/windows

Identical to the first RNN run and to the GLM notebook: same session (Mitt), same 500ms
forward-extending windows.


In [ ]:
session_dir = '../data/raw/080718_mitt'
session_name = '080718_mitt'

bvr = np.load(f'{session_dir}/{session_name}_bvr.npz', allow_pickle=True)
bvr_data = bvr['data']
bvr_keys = bvr['keys'].tolist()

lfp = np.load(f'{session_dir}/{session_name}_lfp.npz', allow_pickle=True)
lfp_data = lfp['data']
lfp_keys = lfp['keys'].tolist()

fs = get_sampling_rate(bvr_data, bvr_keys)
labels = build_labels(bvr_data, bvr_keys)

WINDOW_MS = cfg['data']['window_ms']
windows, kept_idx = segment_trials(lfp_data, labels['trial_idx'], WINDOW_MS, fs)

kept_mask = np.isin(labels['trial_idx'], kept_idx)
inseq_outseq = labels['inseq_outseq'][kept_mask]
odor_id = labels['odor_id'][kept_mask]

print("windows shape:", windows.shape, "-> (n_trials, n_channels, window_samples)")
print("n_trials:", len(kept_idx))


## 3. Reshape for the RNN and split train/val/test

Same trial-level split as before (documented limitation: single-session sanity check only).


In [ ]:
X = windows.transpose(0, 2, 1).astype(np.float32)

n_trials = X.shape[0]
rng = np.random.RandomState(cfg['seed'])
perm = rng.permutation(n_trials)

train_frac = cfg['data']['train_frac']
val_frac = cfg['data']['val_frac']
n_train = int(train_frac * n_trials)
n_val = int(val_frac * n_trials)

train_i = perm[:n_train]
val_i = perm[n_train:n_train + n_val]
test_i = perm[n_train + n_val:]

print(f"train: {len(train_i)}  val: {len(val_i)}  test: {len(test_i)}")


## 4. Standardize

Z-score each channel using ONLY the training set's mean/std.


In [ ]:
train_mean = X[train_i].mean(axis=(0, 1), keepdims=True)
train_std = X[train_i].std(axis=(0, 1), keepdims=True) + 1e-8

X_norm = (X - train_mean) / train_std


## 5. Dataset and DataLoader

In [ ]:
class TrialDataset(Dataset):
    def __init__(self, X, inseq_labels, odor_labels, indices):
        self.X = X[indices]
        self.inseq_labels = inseq_labels[indices]
        self.odor_labels = odor_labels[indices]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return (
            torch.tensor(self.X[i], dtype=torch.float32),
            torch.tensor(self.inseq_labels[i], dtype=torch.long),
            torch.tensor(self.odor_labels[i], dtype=torch.long),
        )

train_ds = TrialDataset(X_norm, inseq_outseq, odor_id, train_i)
val_ds = TrialDataset(X_norm, inseq_outseq, odor_id, val_i)
test_ds = TrialDataset(X_norm, inseq_outseq, odor_id, test_i)

BATCH_SIZE = cfg['train']['batch_size']
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


## 6. Build the model

In [ ]:
n_channels = X.shape[2]

model = MultiTaskRNN(
    input_size=n_channels,
    hidden_size=cfg['model']['hidden_size'],
    num_layers=cfg['model']['num_layers'],
    rnn_type=cfg['model']['type'],
    bidirectional=cfg['model']['bidirectional'],
    dropout=cfg['model']['dropout'],
).to(device)

print(model)


## 7. Compute class weights (THE FIX)

Same idea as `class_weight='balanced'` in the GLM: each class's weight is inversely proportional to how
often it appears in the training set, so the loss penalizes mistakes on rare classes (OutSeq, and
less-common odors) more heavily. This removes the "always guess the majority class" shortcut that
minimized the unweighted loss last time.

Computed from the TRAINING set only, same rule as standardization, never let information from val/test
leak into anything the model or its training process uses.


In [ ]:
def compute_balanced_weights(labels_subset, n_classes):
    counts = np.bincount(labels_subset, minlength=n_classes)
    counts = np.maximum(counts, 1)  # avoid divide-by-zero for any missing class in this split
    weights = len(labels_subset) / (n_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

inseq_train_labels = inseq_outseq[train_i]
inseq_weights = compute_balanced_weights(inseq_train_labels, 2).to(device)
print("InSeq/OutSeq class weights [OutSeq, InSeq]:", inseq_weights.cpu().numpy())

# odor loss is only computed on InSeq trials (per meeting notes), so weights
# should reflect the odor distribution among InSeq training trials specifically
odor_train_mask = (inseq_outseq[train_i] == 1)
odor_train_labels = odor_id[train_i][odor_train_mask]
odor_weights = compute_balanced_weights(odor_train_labels, 5).to(device)
print("Odor class weights [A, B, C, D, E]:", odor_weights.cpu().numpy())


## 8. Train

Same training loop as before, but now using two separate, class-weighted loss functions, one per head.


In [ ]:
ce_loss_inseq = nn.CrossEntropyLoss(weight=inseq_weights)
ce_loss_odor = nn.CrossEntropyLoss(weight=odor_weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg['train']['learning_rate'],
    weight_decay=cfg['train']['weight_decay'],
)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_inseq_loss, total_odor_loss, n_batches = 0.0, 0.0, 0.0, 0

    with torch.set_grad_enabled(train):
        for x, y_inseq, y_odor in loader:
            x, y_inseq, y_odor = x.to(device), y_inseq.to(device), y_odor.to(device)

            inseq_logits, odor_logits = model(x)
            loss_inseq = ce_loss_inseq(inseq_logits, y_inseq)

            odor_mask = (y_inseq == 1)
            if odor_mask.sum() > 0:
                loss_odor = ce_loss_odor(odor_logits[odor_mask], y_odor[odor_mask])
            else:
                loss_odor = torch.tensor(0.0, device=device)

            loss = (cfg['train']['loss_weight_inseq'] * loss_inseq
                    + cfg['train']['loss_weight_odor'] * loss_odor)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_inseq_loss += loss_inseq.item()
            total_odor_loss += loss_odor.item()
            n_batches += 1

    return total_loss / n_batches, total_inseq_loss / n_batches, total_odor_loss / n_batches


train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0
patience = cfg['train']['early_stopping_patience']
best_state = None

for epoch in range(cfg['train']['epochs']):
    train_loss, train_li, train_lo = run_epoch(train_loader, train=True)
    val_loss, val_li, val_lo = run_epoch(val_loader, train=False)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch:2d}  train_loss={train_loss:.4f} (inseq={train_li:.4f} odor={train_lo:.4f})  "
          f"val_loss={val_loss:.4f} (inseq={val_li:.4f} odor={val_lo:.4f})")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
print("\nLoaded best model (lowest validation loss)")


### Visualize training curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_losses, label='train loss')
ax.plot(val_losses, label='val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Combined loss (InSeq/OutSeq + odor)')
ax.set_title('Training curve (class-weighted)')
ax.legend()
plt.tight_layout()
plt.show()


## 9. Evaluate on the held-out test set

In [ ]:
model.eval()
all_inseq_logits, all_odor_logits, all_y_inseq, all_y_odor = [], [], [], []

with torch.no_grad():
    for x, y_inseq, y_odor in test_loader:
        x = x.to(device)
        inseq_logits, odor_logits = model(x)
        all_inseq_logits.append(inseq_logits.cpu())
        all_odor_logits.append(odor_logits.cpu())
        all_y_inseq.append(y_inseq)
        all_y_odor.append(y_odor)

all_inseq_logits = torch.cat(all_inseq_logits)
all_odor_logits = torch.cat(all_odor_logits)
all_y_inseq = torch.cat(all_y_inseq).numpy()
all_y_odor = torch.cat(all_y_odor).numpy()

inseq_preds = all_inseq_logits.argmax(dim=1).numpy()
odor_preds = all_odor_logits.argmax(dim=1).numpy()

bal_acc_inseq = balanced_accuracy_score(all_y_inseq, inseq_preds)
acc_inseq = accuracy_score(all_y_inseq, inseq_preds)
print(f"InSeq/OutSeq  accuracy={acc_inseq:.3f}  balanced_accuracy={bal_acc_inseq:.3f}  (chance=0.500)")

inseq_test_mask = (all_y_inseq == 1)
bal_acc_odor = balanced_accuracy_score(all_y_odor[inseq_test_mask], odor_preds[inseq_test_mask])
acc_odor = accuracy_score(all_y_odor[inseq_test_mask], odor_preds[inseq_test_mask])
print(f"Odor identity accuracy={acc_odor:.3f}  balanced_accuracy={bal_acc_odor:.3f}  (chance=0.200)")
print(f"(evaluated on {inseq_test_mask.sum()} InSeq test trials)")


### Confusion matrices

If the fix worked, the diagonal should no longer be empty, some real spread across both predicted
classes (InSeq/OutSeq) and across more than one predicted odor.


In [ ]:
cm_inseq = confusion_matrix(all_y_inseq, inseq_preds)
cm_odor = confusion_matrix(all_y_odor[inseq_test_mask], odor_preds[inseq_test_mask], labels=[0, 1, 2, 3, 4])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].imshow(cm_inseq, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['OutSeq', 'InSeq'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['OutSeq', 'InSeq'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('InSeq/OutSeq (RNN, weighted)')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm_inseq[i, j], ha='center', va='center',
                      color='white' if cm_inseq[i, j] > cm_inseq.max() / 2 else 'black')

odor_labels = ['A', 'B', 'C', 'D', 'E']
axes[1].imshow(cm_odor, cmap='Blues')
axes[1].set_xticks(range(5)); axes[1].set_xticklabels(odor_labels)
axes[1].set_yticks(range(5)); axes[1].set_yticklabels(odor_labels)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].set_title('Odor identity (RNN, weighted)')
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, cm_odor[i, j], ha='center', va='center',
                      color='white' if cm_odor[i, j] > cm_odor.max() / 2 else 'black')

plt.tight_layout()
plt.show()


## Summary (fill in after running)

Compare against both prior baselines:
- GLM (notebook 03): InSeq/OutSeq balanced accuracy ~0.559, odor balanced accuracy ~0.346
- RNN v1, unweighted (notebook 04 first run): InSeq/OutSeq 0.500 (majority collapse), odor 0.179 (below chance)
- RNN v2, weighted (this notebook): fill in from above

Check specifically: does the InSeq/OutSeq confusion matrix now catch at least some OutSeq trials
(nonzero top-left cell)? Does the odor confusion matrix show predictions spread across more than just
column A? Even a modest, imperfect diagonal here would confirm the fix worked, this isn't about beating
the GLM yet, it's about confirming the RNN is now actually trying to discriminate classes rather than
taking the shortcut.
